# How the Internet Works — APIs for Advocacy

APIs are how software talks to software. Every advocacy tool you'll build — sentiment trackers, product scanners, policy monitors — uses APIs under the hood. Today you'll make your first API calls using tools that are directly relevant to the movement.

By the end of this notebook, you'll be able to:
- Make HTTP requests to real-world APIs
- Parse JSON responses and extract useful data
- Understand the four core HTTP methods (GET, POST, PUT, DELETE)
- See how these skills connect to building tools for animal liberation and climate action

**Before you start: File → Save a copy in Drive**

## Step 1: Is This Product Vegan? (OpenFoodFacts API)

OpenFoodFacts is a free, open database of food products from around the world. Anyone can contribute, and anyone can query it — no API key needed.

It contains ingredient lists, nutrition data, allergen warnings, and labels like "vegan" or "palm oil free" for hundreds of thousands of products.

Imagine building a tool that lets shoppers scan a barcode and instantly know if a product is vegan. That tool would use exactly this API.

Let's start by fetching data for a product using its barcode.

In [ ]:
import requests
import json

try:
    url = "https://world.openfoodfacts.org/api/v2/product/7622210449283.json"
    response = requests.get(url)
    print(f"Status code: {response.status_code}")

    data = response.json()
    # Pretty-print a preview (first ~80 lines) so we can see the structure
    full_text = json.dumps(data, indent=2)
    lines = full_text.split("\n")
    print("\n".join(lines[:80]))
    if len(lines) > 80:
        print(f"\n... ({len(lines) - 80} more lines)")
    print("\u2705 API call successful!")
except Exception as e:
    print(f"Something went wrong: {e}")

## Step 2: Extract the Information We Need

The full response has hundreds of fields — nutrition tables, packaging info, contributor data, and more. We need to navigate the JSON tree and find the fields that matter for advocacy.

JSON is structured like nested dictionaries. To get the product name, we go: `response → product → product_name`. Each arrow is a key lookup.

Let's extract the fields that would power a vegan product checker.

In [ ]:
try:
    product = data.get("product", {})

    name = product.get("product_name", "Unknown")
    ingredients = product.get("ingredients_text", "Not available")
    labels = product.get("labels_tags", [])
    nutriscore = product.get("nutriscore_grade", "Not available")
    num_ingredients = len(product.get("ingredients", []))
    analysis = product.get("ingredients_analysis_tags", [])

    # Determine vegan status from labels
    vegan_status = "Yes" if "en:vegan" in labels else ("No" if "en:non-vegan" in labels else "Unknown")
    # Determine palm oil status from analysis tags
    has_palm_oil = "Yes" if "en:palm-oil" in analysis else ("No" if "en:palm-oil-free" in analysis else "Unknown")

    print(f"Product:        {name}")
    print(f"Ingredients:    {ingredients[:120]}..." if len(ingredients) > 120 else f"Ingredients:    {ingredients}")
    print(f"Labels:         {', '.join(labels[:5])}")
    print(f"Nutri-Score:    {nutriscore}")
    print(f"# Ingredients:  {num_ingredients}")
    print(f"Analysis tags:  {', '.join(analysis)}")
    print(f"\nProduct: {name} | Vegan: {vegan_status} | Palm oil: {has_palm_oil}")
    print("\u2705 Data extracted successfully!")
except Exception as e:
    print(f"Something went wrong: {e}")

> 🎯 **Your Turn:** Try a different product! Find a barcode on a product near you (or search [openfoodfacts.org](https://world.openfoodfacts.org)). Replace the barcode number below and run the cell.

In [ ]:
# Replace this with any barcode you want to look up
my_barcode = "YOUR_BARCODE_HERE"

try:
    url = f"https://world.openfoodfacts.org/api/v2/product/{my_barcode}.json"
    resp = requests.get(url)
    p = resp.json().get("product", {})

    name = p.get("product_name", "Unknown")
    labels = p.get("labels_tags", [])
    analysis = p.get("ingredients_analysis_tags", [])
    vegan = "Yes" if "en:vegan" in labels else ("No" if "en:non-vegan" in labels else "Unknown")
    palm = "Yes" if "en:palm-oil" in analysis else ("No" if "en:palm-oil-free" in analysis else "Unknown")

    print(f"Product: {name} | Vegan: {vegan} | Palm oil: {palm}")
    print("\u2705 Lookup complete!")
except Exception as e:
    print(f"Something went wrong: {e}. Check your barcode and try again.")

> 💡 **What could you build with this?** A browser extension that flags non-vegan products while shopping. A Telegram bot that scans barcodes. A tool that tracks which brands are going vegan. All of these start with the API call you just made.

## Step 3: Understanding HTTP Methods

So far we've used **GET** — the most common HTTP method. It reads data without changing anything. But APIs support four core methods:

| Method | Purpose | Example |
|--------|---------|--------|
| **GET** | Read data | Fetch a product's ingredients |
| **POST** | Create new data | Submit an advocacy report |
| **PUT** | Update existing data | Edit a campaign record |
| **DELETE** | Remove data | Delete a draft post |

When you build advocacy tools, you'll need all of these — GET to read data, POST to submit reports, PUT to update records, DELETE to remove content.

Let's practice POST using a safe, free practice API.

In [ ]:
# JSONPlaceholder is a free fake API for testing — nothing is actually saved
try:
    report = {
        "title": "Factory Farm Emissions Report",
        "body": "Livestock accounts for 14.5% of global greenhouse gas emissions according to the FAO.",
        "userId": 1
    }
    response = requests.post("https://jsonplaceholder.typicode.com/posts", json=report)

    print(f"Status code: {response.status_code}")  # 201 = Created
    print(f"Response: {json.dumps(response.json(), indent=2)}")
    # JSONPlaceholder fakes the write — nothing is actually saved.
    # But real APIs work the same way.
    print("\u2705 POST request successful! (201 = resource created)")
except Exception as e:
    print(f"Something went wrong: {e}")

## Step 4: Environmental Data

The same skills — making requests, parsing JSON, extracting fields — work for climate advocacy too. Environmental datasets track emissions, deforestation, water usage, and more.

In a real project, this data would come from an API (like the World Bank Climate API or FAOSTAT). For now, we're practicing reading JSON structures with real advocacy data sourced from peer-reviewed research.

In [ ]:
# Real advocacy data structured as JSON — sourced from FAO, IPCC, and published research
environmental_data = json.loads('''
{
  "title": "Animal Agriculture Environmental Impact",
  "sources": ["FAO - Tackling Climate Change Through Livestock (2013)", "IPCC AR6 (2022)", "Poore & Nemecek, Science (2018)"],
  "livestock_emissions": {
    "total_share_of_global_ghg": "14.5%",
    "by_species": {
      "cattle_beef": "41%",
      "cattle_milk": "20%",
      "pigs": "9%",
      "poultry_and_eggs": "8%",
      "small_ruminants": "6%",
      "other": "16%"
    }
  },
  "land_use": {
    "livestock_share_of_agricultural_land": "77%",
    "livestock_share_of_global_calories": "18%",
    "livestock_share_of_global_protein": "37%",
    "amazon_deforestation_for_cattle": "80%"
  },
  "water_usage_liters_per_kg": {
    "beef": 15415,
    "pork": 5988,
    "chicken": 4325,
    "tofu": 2523,
    "lentils": 1250,
    "vegetables": 322
  }
}
''')

try:
    emissions = environmental_data["livestock_emissions"]
    land = environmental_data["land_use"]
    water = environmental_data["water_usage_liters_per_kg"]

    print(f"Livestock's share of global GHG emissions: {emissions['total_share_of_global_ghg']}")
    print(f"Largest contributor: Beef cattle at {emissions['by_species']['cattle_beef']} of livestock emissions")
    print(f"\nLand use paradox: {land['livestock_share_of_agricultural_land']} of farmland → only {land['livestock_share_of_global_calories']} of calories")
    print(f"Amazon deforestation driven by cattle ranching: {land['amazon_deforestation_for_cattle']}")

    print(f"\nWater usage (liters per kg):")
    for food, liters in sorted(water.items(), key=lambda x: x[1], reverse=True):
        bar = "#" * (liters // 500)
        print(f"  {food:<12} {liters:>6}L  {bar}")

    print(f"\nSources: {', '.join(environmental_data['sources'])}")
    print("\u2705 Environmental data parsed successfully!")
except Exception as e:
    print(f"Something went wrong: {e}")

> 🎯 **Bonus Challenge:** The OpenFoodFacts API also supports search. Try the URL below to search for oat milk products:
>
> `https://world.openfoodfacts.org/cgi/search.pl?search_terms=oat+milk&search_simple=1&action=process&json=1`
>
> How many oat milk products are in the database? What brands appear most often?

In [ ]:
# Try searching for oat milk — or replace with your own search term
search_term = "oat milk"  # Change this to search for something else

try:
    search_url = f"https://world.openfoodfacts.org/cgi/search.pl?search_terms={search_term.replace(' ', '+')}&search_simple=1&action=process&json=1"
    resp = requests.get(search_url)
    results = resp.json()

    total = results.get("count", 0)
    products = results.get("products", [])
    print(f"Total '{search_term}' products in database: {total}")
    print(f"Showing first {len(products)} results:\n")

    for p in products[:8]:
        name = p.get("product_name", "Unnamed")
        brand = p.get("brands", "Unknown brand")
        print(f"  {brand}: {name}")
    print("\u2705 Search complete!")
except Exception as e:
    print(f"Something went wrong: {e}")

> 💡 **What just happened?** You queried a product database to check if food is vegan, explored environmental emissions data, and practiced all the core HTTP methods. Every advocacy tool — from vegan product scanners to deforestation monitors to corporate accountability trackers — is built on exactly these foundations. The internet is infrastructure. APIs are the building blocks. You now know how to use them.